# 09 Broadcast and Join Strategies

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Know Spark's five join strategies (broadcast hash, sort-merge, shuffled hash, broadcast nested loop, cartesian), how Spark chooses between them, how to steer the choice with hints, and when broadcasting is a bad idea.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"Broadcast join vs sort-merge join?" is one of the six spoken answers in this learning path, and joins are the most common reason a Spark job is slow. Choosing (or letting AQE choose) the right strategy can turn a 40-minute join into a 2-minute one.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The five strategies</b><br>
| Strategy | How it works | When it's used | Shuffle? |<br>|---|---|---|---|<br>| <b>Broadcast hash join (BHJ)</b> | Small side is collected to the driver and sent to every executor, which builds a hash table. Big side is streamed through it | One side is small (below <code>autoBroadcastJoinThreshold</code>, 10 MB by default, or hinted), equi-join | No shuffle of the big side |<br>| <b>Sort-merge join (SMJ)</b> | Both sides are shuffled by the key, sorted, then merged | Default for large equi-joins | Both sides |<br>| <b>Shuffled hash join (SHJ)</b> | Both sides are shuffled, and a hash table is built from the smaller side per partition (no sort) | Hinted, or chosen when one side is much smaller but too big to broadcast | Both sides |<br>| <b>Broadcast nested loop join (BNLJ)</b> | Small side broadcast, every pair compared with the condition | Non-equi joins (<code>&lt;</code>, <code>between</code>, <code>!=</code>) with a small side | No |<br>| <b>Cartesian product</b> | Every row with every row | Cross joins, non-equi joins of two large sides | Yes, very expensive |
</div>

```
BROADCAST HASH JOIN                          SORT-MERGE JOIN
 small ──collect──▶ driver ──broadcast──┐     big A ──shuffle by key──▶ sort ─┐
                                        ▼                                       ├─ merge
 big partition 1 ──▶ [hash table] ──▶ out     big B ──shuffle by key──▶ sort ─┘
 big partition 2 ──▶ [hash table] ──▶ out
 (big side never moves)                       (both sides move and sort)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retailer joins 3 billion sales rows to a 40,000-row product table. Spark's estimate of the product table's size was wrong, because it was computed from a complex subquery, so it chose a sort-merge join that shuffled 900 GB. A <code>broadcast()</code> hint on the product side removed the shuffle of the fact table, and the job went from 50 minutes to 6. On Databricks, AQE now often makes this switch automatically at runtime, once it sees the real size.
</div>

## Syntax

```python
from pyspark.sql import functions as F

big.join(F.broadcast(small), "key")              # broadcast hint
big.join(small.hint("broadcast"), "key")         # same
big.join(other.hint("merge"), "key")             # prefer sort-merge
big.join(other.hint("shuffle_hash"), "key")      # prefer shuffled hash
```

```sql
SELECT /*+ BROADCAST(p) */ * FROM sales s JOIN products p ON s.product_id = p.product_id;
SELECT /*+ MERGE(p) */ ...;   SELECT /*+ SHUFFLE_HASH(p) */ ...;
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a large fact table, a small dimension, and a medium table, plus a helper that names the join strategy in a plan.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
import io, contextlib, re, time
from pyspark.sql import functions as F

def plan_text(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

def strategy(df):
    text = plan_text(df)
    for op in ["BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin", "BroadcastNestedLoopJoin", "CartesianProduct"]:
        if op in text:
            return op
    return "no join"

sales = spark.range(0, 5_000_000, 1, numPartitions=8).select(
    F.col("id").alias("sale_id"), (F.col("id") % 10_000).alias("product_id"), (F.col("id") % 997).alias("amount")
)
products = spark.range(10_000).select(F.col("id").alias("product_id"), (F.col("id") % 20).alias("category"))
customers = spark.range(2_000_000).select(F.col("id").alias("customer_id"), (F.col("id") % 50).alias("segment"))

## 1. What Spark chooses by itself

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins sales to the small products table without hints and prints the chosen strategy.<br><br>
<b>Why it matters</b><br>For in-memory or well-sized sources, Spark estimates the products table as small and broadcasts it automatically. It uses size estimates from the plan (and table statistics for tables).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>BroadcastHashJoin</code>.
</div>

In [0]:
print(strategy(sales.join(products, "product_id")))

## 2. Steering with hints

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the same join with each hint and prints the strategy and time.<br><br>
<b>Why it matters</b><br>Hints override the optimizer's choice. Use them when you know better than its estimates, and check the plan afterwards: Spark may ignore a hint that can't apply (for example, broadcasting the preserved side of an outer join).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The broadcast hint gives <code>BroadcastHashJoin</code>, <code>merge</code> gives <code>SortMergeJoin</code>, and <code>shuffle_hash</code> gives <code>ShuffledHashJoin</code>. On data this small the broadcast and shuffled-hash timings are close, and sort-merge is usually the slowest because it sorts both sides. The gap in favour of broadcasting grows with the size of the big side, because it's the only strategy that doesn't shuffle it.
</div>

In [0]:
sales.join(products, "product_id").write.format("noop").mode("overwrite").save()   # warm-up run

for hint in ["broadcast", "merge", "shuffle_hash"]:
    joined = sales.join(products.hint(hint), "product_id").groupBy("category").agg(F.sum("amount"))
    start = time.time()
    joined.write.format("noop").mode("overwrite").save()
    print(f"{hint:<13} -> {strategy(joined):<20} {time.time() - start:.2f}s")

## 3. Two large tables: sort-merge join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins sales to the 2-million-row customers table (no broadcast) and prints the plan.<br><br>
<b>Why it matters</b><br>When neither side is small, sort-merge is the default. Both sides are shuffled by the key and sorted. That's the expensive case, and it's why you filter and select columns <b>before</b> big joins.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>SortMergeJoin</code> with an <code>Exchange hashpartitioning</code> and a <code>Sort</code> on each side.
</div>

In [0]:
sales_c = sales.withColumn("customer_id", F.col("sale_id") % 2_000_000)
big_join = sales_c.join(customers.hint("merge"), "customer_id")
print(strategy(big_join))
big_join.explain()

## 4. Non-equi joins: nested loop and cartesian

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins sales to price bands with a range condition (<code>between</code>), and shows the strategy for a small and a large right side.<br><br>
<b>Why it matters</b><br>Hash and sort-merge joins need an <b>equality</b> condition. Range or inequality conditions fall back to nested loop or cartesian joins, which compare many pairs. With a small side it's fine (BNLJ). With two large sides it can run for hours.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>BroadcastNestedLoopJoin</code> for both: the bands table is tiny, and the second query's 1,000-row side is still small enough to broadcast. With two genuinely large sides, Spark would have to use a <code>CartesianProduct</code>, comparing every row with every row.
</div>

In [0]:
bands = spark.createDataFrame([(0, 99, "low"), (100, 499, "mid"), (500, 999, "high")], "lo INT, hi INT, band STRING")
range_join = sales.join(bands, (sales.amount >= bands.lo) & (sales.amount <= bands.hi))
print("small side, range condition:", strategy(range_join))

s1k = sales.limit(1000)
big_range = s1k.join(customers.hint("merge"), s1k.sale_id < customers.customer_id)
print("two sides, range condition :", strategy(big_range))

## 5. When broadcasting is a bad idea

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the limits of broadcasting and shows which side of an outer join can be broadcast.<br><br>
<b>Why it matters</b>
<ul><li>The broadcast table is <b>collected on the driver</b> and copied to every executor. A "small" table of several GB can crash the driver or use a lot of executor memory. Spark limits broadcasts to 8 GB, but problems start much earlier</li><li>For a <b>left outer join</b>, only the <b>right</b> side can be broadcast (the preserved left side must be streamed). For a right outer join, only the left. For a full outer join, neither</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For a left join with the hint on the right side: <code>BroadcastHashJoin</code>. With the hint on the left (preserved) side: Spark can't honour it and uses <code>SortMergeJoin</code>.
</div>

In [0]:
small_left = products.hint("broadcast")
print("left join, hint on right side:", strategy(sales.join(F.broadcast(products), "product_id", "left")))
print("left join, hint on left side :", strategy(small_left.join(sales, "product_id", "left")))

## 6. AQE can switch strategies at runtime

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins sales to a <b>filtered</b> customers table. The optimizer's static estimate is large, but after filtering the real size is tiny.<br><br>
<b>Why it matters</b><br>Static estimates are often wrong after filters, aggregations or UDFs. With AQE, Spark first runs the shuffle stages, measures the actual sizes, and converts a planned sort-merge join into a broadcast join if one side turns out small. The initial plan (in <code>explain</code>) shows <code>SortMergeJoin</code>, and the final plan (Spark UI / query profile) shows <code>BroadcastHashJoin</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The initial plan names <code>SortMergeJoin</code> under an <code>AdaptiveSparkPlan isFinalPlan=false</code>. When run, AQE may switch it. Check the query profile on Databricks to see the final strategy.
</div>

In [0]:
tiny_after_filter = customers.filter(F.col("segment") == 7).filter(F.col("customer_id") < 2000)
adaptive = sales_c.join(tiny_after_filter, "customer_id")
print("initial plan strategy:", strategy(adaptive))
start = time.time()
adaptive.write.format("noop").mode("overwrite").save()
print(f"ran in {time.time() - start:.2f}s (see the final plan in the query profile)")

## Under the hood

```
JoinSelection (planner), for an equi-join:
  1. hints first (BROADCAST > MERGE > SHUFFLE_HASH > SHUFFLE_REPLICATE_NL)
  2. one side's estimated size < autoBroadcastJoinThreshold (10 MB)  -> BroadcastHashJoin
  3. preferSortMergeJoin (default true)                              -> SortMergeJoin
  4. otherwise ShuffledHashJoin when one side is much smaller
Non-equi join: BroadcastNestedLoopJoin if one side can be broadcast, else CartesianProduct

At runtime (AQE): after shuffle stages finish, re-plan with real sizes:
  SortMergeJoin + small side  -> BroadcastHashJoin
  skewed partitions           -> split (skew join)
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> sort-merge and shuffled hash joins shuffle both sides.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> for the big side of a broadcast hash join.

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>spark.sql.autoBroadcastJoinThreshold</code> can't be changed on serverless. Use hints instead, and let AQE (always on there) adjust at runtime.
</div>

In [0]:
spark.sql("SELECT 1").createOrReplaceTempView("dummy")
sales.createOrReplaceTempView("sales_v")
products.createOrReplaceTempView("products_v")
spark.sql("""
    SELECT /*+ BROADCAST(p) */ p.category, sum(s.amount) AS total
    FROM sales_v s JOIN products_v p ON s.product_id = p.product_id
    GROUP BY p.category
""").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: a big join shuffles a huge fact table</b><br>
The small side wasn't recognized as small. Add a broadcast hint, or filter and select first.<br><br>
<b>⛔ Problem: driver out of memory, or "broadcast timeout"</b><br>The broadcast side is too big. Remove the hint and let Spark use sort-merge.<br><br>
**⛔ Problem: the join runs forever and the plan shows <code>CartesianProduct</code>**<br>A missing or non-equi condition between two large tables. Add an equality condition, or pre-bucket ranges into an equi-join key.<br><br>
<b>⛔ Problem: the hint is ignored</b><br>It can't apply, for example broadcasting the preserved side of an outer join. Check <code>explain()</code>.<br><br>
<b>⛔ Problem: one task in the join stage is very slow</b><br>Key skew. Let AQE skew join handle it, or salt the key (<code>05_optimization</code>).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Broadcast join vs sort-merge join?</b><br>
A broadcast hash join sends a copy of the small table to every executor, so the large table is joined in place with no shuffle. It's very fast, but only if the small side fits in memory. A sort-merge join shuffles both sides by the join key, sorts them, and merges them. It scales to any size, but pays for two shuffles and sorts. Spark broadcasts automatically below <code>autoBroadcastJoinThreshold</code> (10 MB), or when hinted, or when AQE detects a small side at runtime.<br><br>

<b>🎤 2. What join strategies does Spark have?</b><br>
Broadcast hash, sort-merge, shuffled hash, broadcast nested loop, and cartesian product. The first three need an equi-join condition. The last two handle non-equi conditions.<br><br>

<b>🎤 3. How do you force a broadcast join?</b><br>
With <code>F.broadcast(df)</code> or <code>df.hint("broadcast")</code> in PySpark, or <code>/*+ BROADCAST(t) */</code> in SQL. Then I verify <code>BroadcastHashJoin</code> in the plan.<br><br>

<b>🎤 4. When should you not broadcast?</b><br>
When the small side is large (hundreds of MB to GB), because it's collected on the driver and copied to every executor. Also, the preserved side of an outer join can't be broadcast.<br><br>

<b>🎤 5. How does AQE affect join strategy?</b><br>
After the shuffle stages run, AQE knows the real sizes. It can convert a planned sort-merge join into a broadcast join if one side turns out small, and split skewed partitions in sort-merge joins.<br><br>

<b>🎤 6. What happens with a join on a <code>&lt;</code> or <code>between</code> condition?</b><br>
Spark can't use hash or sort-merge, so it uses a broadcast nested loop join if one side is small, or a cartesian product otherwise, which compares every pair and can be extremely slow.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A query joins a 2 TB fact table to a 5 MB dimension table, and the Spark UI shows a large shuffle of the fact table. What should the data engineer do?</b><br>
A. Increase <code>spark.sql.shuffle.partitions</code><br>
B. Add a broadcast hint on the dimension table<br>
C. Repartition the fact table by the join key<br>
D. Convert the join to a cross join<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Broadcasting the small side removes the shuffle of the fact table.</details><br><br>

<b>Q2. Which join strategy is Spark's default for joining two large tables on an equality condition?</b><br>
A. Broadcast hash join<br>
B. Sort-merge join<br>
C. Broadcast nested loop join<br>
D. Cartesian product<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Join <code>sales</code> to <code>products</code> and compute total amount per category three times, with the broadcast, merge and shuffle_hash hints. Record the strategies and times on your compute</li><li>Write the broadcast version in SQL with a hint</li><li>Create a non-equi join between <code>sales</code> (limited to 1,000 rows) and <code>bands</code>, and name the strategy</li><li>In two sentences, prepare your spoken answer to "broadcast join vs sort-merge join"</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Three hints, measured
for hint in ["broadcast", "merge", "shuffle_hash"]:
    q = sales.join(products.hint(hint), "product_id").groupBy("category").agg(F.sum("amount").alias("total"))
    start = time.time(); q.collect()
    print(f"{hint:<13} {strategy(q):<20} {time.time() - start:.2f}s")

# 2. SQL hint
print(strategy(spark.sql(
    "SELECT /*+ BROADCAST(p) */ p.category, sum(s.amount) FROM sales_v s JOIN products_v p USING (product_id) GROUP BY p.category"
)))

# 3. Range condition -> nested loop
s1k = sales.limit(1000)
print(strategy(s1k.join(bands, s1k.amount.between(bands.lo, bands.hi))))

# 4. A broadcast hash join copies the small table to every executor so the big table is joined in place
#    with no shuffle; a sort-merge join shuffles and sorts both sides by the key, which scales to any size but costs more.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>BHJ: small side broadcast, no shuffle of the big side. Fastest when one side is small</li><li>SMJ: both sides shuffled and sorted. The default for large equi-joins</li><li>SHJ: both shuffled, hash table per partition, no sort</li><li>BNLJ / cartesian: non-equi joins. Dangerous with two large sides</li><li>Broadcast threshold is 10 MB by default (fixed on serverless). Use hints, and AQE switches at runtime</li><li>Outer joins can only broadcast the non-preserved side</li><li>Always confirm the strategy in <code>explain()</code> or the query profile</li></ul>
</div>

| Want | Code |
|---|---|
| Broadcast | `big.join(F.broadcast(small), "k")` |
| Sort-merge | `a.join(b.hint("merge"), "k")` |
| Shuffled hash | `a.join(b.hint("shuffle_hash"), "k")` |
| SQL hint | `SELECT /*+ BROADCAST(t) */ ...` |
| Check | `explain()` → `BroadcastHashJoin` / `SortMergeJoin` |

## Git commit

```
git add 02_spark_internals/09_broadcast_and_join_strategies.ipynb
git commit -m "add 02_09 broadcast and join strategies notebook"
```